In [4]:
import pandas as pd
import mlflow
from storesales.pipeline import run_experiment

In [ ]:
# MAIN PARAMETERS
ACTIVE_MODEL = "linear_regression"

# linear_regression, xgboost

NUMERIC_SKEWED_FEATURES = [
    "sales_lag15",
    "sales_lag16",
    "sales_lag17",
    "sales_lag18",
    "sales_lag19",
    "sales_lag20",
    "sales_lag21",
    "sales_lag30",
    "sales_lag60",
    "sales_lag364",
    "sales_lag365",
]
NUMERIC_NORMAL_FEATURES = []

TIER_FEATURES = []
CATEGORICAL_FEATURES = []
TARGET_ENCODER = False

TARGET = "sales"

COLUMNS_TO_KEEP = ["date", "family", "store_nbr", "is_int_sales"]

CUTOFFS_STR = ["2017-07-15", "2017-04-15", "2017-01-15", "2016-09-15"]
CUTOFFS_DATETIME = pd.to_datetime(CUTOFFS_STR)

HORIZON = 15

ISSUE_NAME = "lags"
VERSION_NAME = "[15-21]+30,60,364,365"

PATH_TRAIN_DF = "../data/processed/df_train.parquet"

In [10]:
df_results, trained_models = run_experiment(
    model_name=ACTIVE_MODEL,
    skewed_feats=NUMERIC_SKEWED_FEATURES,
    normal_feats=NUMERIC_NORMAL_FEATURES,
    tier_feats=TIER_FEATURES,
    cat_feats=CATEGORICAL_FEATURES,
    target_encoder=TARGET_ENCODER,
    target=TARGET,
    cols_to_keep=COLUMNS_TO_KEEP,
    cutoffs=CUTOFFS_DATETIME,
    horizon=HORIZON,
    issue_name=ISSUE_NAME,
    version_name=VERSION_NAME,
    df_path=PATH_TRAIN_DF,
)

/Users/silviadeanituaaspiazu/Library/Caches/pypoetry/virtualenvs/storesales-timeseriesforecasting-5H9Z9Y-8-py3.15/lib/python3.15/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/Users/silviadeanituaaspiazu/Library/Caches/pypoetry/virtualenvs/storesales-timeseriesforecasting-5H9Z9Y-8-py3.15/lib/python3.15/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing va

Cutoff: 2017-07-15, MAE: 474784.35, RMSE:14469218.30, RMSLE:2.07


2026/11/16 12:32:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/16 12:32:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Cutoff: 2017-04-15, MAE: 482404773.86, RMSE:74511885894.32, RMSLE:2.12


2026/11/16 12:32:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/16 12:32:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Cutoff: 2017-01-15, MAE: 7220263.57, RMSE:367853854.93, RMSLE:2.13


2026/11/16 12:32:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/16 12:32:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Cutoff: 2016-09-15, MAE: 344534.24, RMSE:8483051.80, RMSLE:2.09


2026/11/16 12:32:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/16 12:32:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


GLOBAL RMSLE: 2.1023 GLOBAL BIAS:  122610656.0951


cuales on los splits que he hecho entonces en lo de las familias?

In [4]:
mlflow.end_run()

In [12]:
import mlflow

# 1. Obtenemos el dataframe de MLflow
runs_df = mlflow.search_runs(experiment_names=["Store_Sales_Forecasting"])

# 2. Comprobamos si el dataframe está vacío antes de buscar
if runs_df.empty:
    print("El experimento no tiene ejecuciones registradas todavía.")
else:
    # 3. Buscamos la columna correcta dependiendo de tu versión de MLflow
    columna_nombre = (
        "run_name" if "run_name" in runs_df.columns else "tags.mlflow.runName"
    )

    if columna_nombre in runs_df.columns:
        # Aquí ya puedes hacer tu filtro o visualización sin miedo al KeyError
        print(runs_df[["run_id", columna_nombre, "status"]])
    else:
        print(f"Columnas disponibles: {runs_df.columns.tolist()}")

                             run_id tags.mlflow.runName    status
0  df79cf4bea5d4a32b79fe7106637734c      xgboost_test_1  FINISHED
1  5bef66b7f8e24fa3aaae7a50b61c0d65      xgboost_test_1    FAILED


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Asegurarnos de que hay datos y tenemos la columna del nombre
if not runs_df.empty and columna_nombre in runs_df.columns:

    # 1. Filtrar solo las ejecuciones que terminaron correctamente
    df_valid = runs_df[runs_df["status"] == "FINISHED"].copy()

    # 2. Definir qué métrica quieres comparar.
    # MLflow guarda las métricas con el prefijo 'metrics.'
    # Cámbialo por "metrics.mae", "metrics.wmape" u otra que hayas registrado
    metrica_objetivo = "metrics.rmse"

    if metrica_objetivo in df_valid.columns:
        # 3. Ordenar de menor a mayor error (los mejores modelos arriba)
        # Si tu métrica fuera R2 (donde mayor es mejor), pon ascending=False
        df_sorted = df_valid.sort_values(by=metrica_objetivo, ascending=True)

        # 4. Configurar y crear el gráfico
        plt.figure(figsize=(10, 6))
        ax = sns.barplot(
            data=df_sorted,
            x=metrica_objetivo,
            y=columna_nombre,
            hue=columna_nombre,  # Asignar hue para evitar warnings en versiones recientes de seaborn
            palette="viridis",
            legend=False,
        )

        # 5. Estética del gráfico
        nombre_limpio = metrica_objetivo.replace("metrics.", "").upper()
        plt.title(
            f"Comparación de Modelos (Store Sales) - {nombre_limpio}",
            fontsize=14,
            pad=15,
        )
        plt.xlabel(nombre_limpio, fontsize=12)
        plt.ylabel("Nombre del Experimento / Modelo", fontsize=12)

        # Añadir cuadrícula vertical para facilitar la lectura de los valores
        plt.grid(axis="x", linestyle="--", alpha=0.7)

        # Opcional: Añadir el valor numérico exacto a la derecha de cada barra
        for p in ax.patches:
            ax.annotate(
                f"{p.get_width():.4f}",
                (p.get_width(), p.get_y() + p.get_height() / 2.0),
                ha="left",
                va="center",
                xytext=(5, 0),
                textcoords="offset points",
                fontsize=10,
            )

        plt.tight_layout()
        plt.show()

    else:
        print(f"Error: La métrica '{metrica_objetivo}' no existe en el DataFrame.")
        metricas_disp = [col for col in df_valid.columns if col.startswith("metrics.")]
        print(f"Métricas disponibles para plotear: {metricas_disp}")
else:
    print(
        "No se puede generar el gráfico porque el DataFrame está vacío o falta la columna de nombres."
    )

In [ ]:
import statsmodels.api as sm
import numpy as np
import pandas as pd

cutoff_date = "2017-08-01"
df_eda = df[df["date"] < cutoff_date].copy()

# 2. Separar target y variables
y_train_eda = np.log1p(df_eda["sales"])
X_train_eda = df_eda[skewed_feats].copy()

# 3. Transformaciones rápidas al estilo Pandas (ideal para Statsmodels)
# Logaritmo a las asimétricas
for col in skewed_feats:
    X_train_eda[col] = np.log1p(X_train_eda[col])

# Variables categóricas: get_dummies es mil veces más fácil de leer que OneHotEncoder
X_train_eda = pd.get_dummies(X_train_eda, drop_first=True, dtype=int)

# 4. Ajustar modelo OLS
X_train_sm = sm.add_constant(X_train_eda)
modelo_sm = sm.OLS(y_train_eda, X_train_sm).fit()

# Ver resultados
print(modelo_sm.summary())